<a id="unit-u.799f43982ee24c24a9963575b86600f2"></a>

<div>
  <div>06-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Apache Doris жүйесінде деректерді біріктіру</div>
  <p>Нәтиже сақтауы тиіс жолдарға қарай Join таңдап, көзделген нәтиже түйіршіктілігін қорғаңыз және логикалық Join мағынасын Doris физикалық орындауымен байланыстырыңыз.</p>
  <span>INNER · OUTER · SEMI · ANTI · Hash · Nested Loop · Broadcast · Bucket Shuffle · Runtime Filter</span>
</div>

**6-модуль: Apache Doris жүйесінде деректерді біріктіру** · [Сабақ](course.md) · [Тест](quiz06_join_semantics_and_execution.ipynb)

Шамамен 30 минут жоспарлаңыз.

Бұл зертханалық жұмыс түйіршіктілігі әр бастапқы оқиғаға бір жол болатын `events_modelled` кестесімен жалғасады. Өзінің детерминді өлшемін және шағын Join жағдайларын жасайды, сондықтан қосымша ашық деректер жиыны, S3 нысаны немесе тіркелгі деректері қажет емес.

<a id="unit-u.48399e9e2396422788ef8ad58a192c91"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

Келесі ұяшықты 1-бөлімге дейін орындаңыз. Ол ортақ курс көмекшісін жүктеп, кейінгі әр ұяшық пайдаланатын `lab` нысанын жасайды. Jupyter ядросын қайта іске қосқаннан кейін оны қайта орындаңыз. Ол Docker іске қоспайды және Doris деректерін өзгертпейді.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


<a id="unit-u.fb1f4bb06815418593b04aa2cd54bc99"></a>

<a id="1-build-a-controlled-product-relationship-from-the-baseline"></a>

## 1. Бастапқы жиыннан бақыланатын өнім байланысын құру

Детерминді `dim_products` кестесі бастапқы өнім идентификаторларынан жергілікті жасалады және сәйкес емес аз ғана бақыланатын сынақ жолдарын қамтиды. Екінші ашық деректер жиыны немесе сыртқы жүктеу қажет емес.

Кесте Unique Key моделін қолданып, әр өнімге бір жол сақтайды. Санат пен бренд — `product_id` мәнінен есептелетін детерминді оқу атрибуттары. Онда 204,231 өнім жолы бар; төмендегі үш идентификатор кейін толық өлшем ретінде емес, тек бақыланатын мысалдар ретінде таңдалады:

| `product_id` | `events_modelled` ішінде | `dim_products` ішінде | Байланысы |
|---:|---|---|---|
| `1004767` | Иә | Иә | Сәйкес |
| `1005115` | Иә | Жоқ | Тек оқиғада |
| `999999999` | Жоқ | Иә | Тек өлшемде |

`1005115` өнімі әдейі түсіріліп қалдырылады, ал `999999999` өнімі әдейі қосылады. Бұл бақыланатын ерекшеліктер сәйкес емес байланыстың екі бағытын да көрінетін етеді.


In [ ]:
lab.start_container()
lab.connect()
lab.execute("USE doris_course")
lab.check_prerequisites(6)
lab.execute("DROP TABLE IF EXISTS dim_products")
lab.execute("""
CREATE TABLE dim_products (
    product_id BIGINT NOT NULL,
    category VARCHAR(32) NOT NULL,
    brand VARCHAR(32) NOT NULL
)
UNIQUE KEY(product_id)
DISTRIBUTED BY HASH(product_id) BUCKETS 4
PROPERTIES ("replication_num" = "1")
""")

lab.insert("""
INSERT INTO dim_products (product_id, category, brand)
SELECT DISTINCT
    product_id,
    CONCAT('category_', MOD(product_id, 8) + 1),
    CONCAT('brand_', MOD(product_id, 20) + 1)
FROM events_modelled
WHERE product_id <> 1005115
UNION ALL
SELECT 999999999, 'category_unused', 'brand_unused'
""", title="Build the deterministic product dimension");

<a id="unit-u.2764f431e1164ad1858533d91833c696"></a>

**Күтілетін нәтиже:** алдымен жасыл белгі 4-зертханалық жұмыстың `events_modelled` кестесінде 10,158,080 жол барын растайды. Содан кейін INSERT 204,231 әсер етілген жолды хабарлайды. `dim_products` ішінде әр `product_id` бір рет кездеседі, сондықтан бір сәйкес оқиға жолы ең көбі бір санат пен бренд ала алады. Ұяшықты қайта орындау кестені жойып, қайта құрады, сондықтан жолдарды қайталамайды. Жоғарыдағы бақыланатын байланыстарға келесі бөлімде сұрау жасалады.

<a id="unit-u.5972bfee90f44b7cbe7b5b8d60663348"></a>

<a id="2-choose-the-join-type-from-the-rows-the-result-must-retain"></a>

## 2. Нәтиже сақтауы тиіс жолдарға қарай Join түрін таңдау

Келесі сұраулар сақталатын жолдар оңай көрінуі үшін оқиға кірісін екі өнім идентификаторымен әдейі шектейді. Бұл `WHERE` тармағы `dim_products` мазмұнын шектемейді: бақыланатын салыстыру ретінде сәйкес `1004767` өнімін және тек оқиғада бар `1005115` өнімін таңдайды.

`INNER JOIN` тек сәйкес жұптарды сақтайды. `LEFT OUTER JOIN` өнім анықтамасы жоқ оқиға жолдарын да сақтап, оң жақ атрибуттарды `NULL` толтырады.


In [ ]:
lab.sql("""
SELECT e.product_id, d.category, COUNT(*) AS retained_event_rows
FROM events_modelled e
INNER JOIN dim_products d ON e.product_id = d.product_id
WHERE e.product_id IN (1004767, 1005115)
GROUP BY e.product_id, d.category
ORDER BY e.product_id
""", title="Rows retained by INNER JOIN")

lab.sql("""
SELECT e.product_id, d.category, COUNT(*) AS retained_event_rows
FROM events_modelled e
LEFT OUTER JOIN dim_products d ON e.product_id = d.product_id
WHERE e.product_id IN (1004767, 1005115)
GROUP BY e.product_id, d.category
ORDER BY e.product_id
""", title="Rows retained by LEFT OUTER JOIN");


<a id="unit-u.0fff58574b364401b0c5000b51a414b6"></a>

`LEFT SEMI JOIN` және `LEFT ANTI JOIN` тек сақталатын сол жақ бағандарын қайтарады. Сондықтан сұраулар оң жақ атрибуттардың орнына сақталған `product_id` және оқиға санын көрсетеді. Semi сәйкес өнімді, Anti тек оқиғада бар өнімді сақтайды.


In [ ]:
lab.sql("""
SELECT e.product_id, COUNT(*) AS retained_event_rows
FROM events_modelled e
LEFT SEMI JOIN dim_products d ON e.product_id = d.product_id
WHERE e.product_id IN (1004767, 1005115)
GROUP BY e.product_id
""", title="Rows retained by LEFT SEMI JOIN")

lab.sql("""
SELECT e.product_id, COUNT(*) AS retained_event_rows
FROM events_modelled e
LEFT ANTI JOIN dim_products d ON e.product_id = d.product_id
WHERE e.product_id IN (1004767, 1005115)
GROUP BY e.product_id
""", title="Rows retained by LEFT ANTI JOIN");


<a id="unit-u.4b14d2f1fcaa4ca6a325bc3bb18173d6"></a>

**Күтілетін нәтиже:** әр қысқа сұрау сақтау ережесін көрінетін етеді:

- `INNER JOIN` тек `1004767` қайтарады: `category_8` санаты және сақталған 105,046 оқиға жолы. `1005115` көрінбейді, өйткені сәйкес өлшем жолы жоқ.
- `LEFT OUTER JOIN` екі идентификаторды да қайтарады. `1005115` 89,522 оқиға жолын сақтайды, ал санаты `NULL`.
- `LEFT SEMI JOIN` тек сәйкес `1004767` идентификаторын және оның 105,046 оқиға жолын қайтарады.
- `LEFT ANTI JOIN` тек оқиғада бар `1005115` идентификаторын және оның 89,522 оқиға жолын қайтарады.

Осы бақыланатын кіріс үшін Semi және Anti бірін-бірі толықтырады: бірі сәйкес ішкі жиынды, екіншісі сәйкес емес ішкі жиынды сақтайды. RIGHT және FULL OUTER JOIN басқа жақтан тиісті сақтау ережелерін қолданады. CROSS JOIN шартсыз және декарт көбейтіндісін береді, сондықтан бастапқы жиында орындалмайды.


<a id="unit-u.604d64f43e944ddcb313b24b2ca032c8"></a>

<a id="3-produce-a-fact-dimension-analytical-result"></a>

## 3. Факт-өлшем талдамалық нәтижесін жасау

Негізгі талдамалық сұрау енді сатып алу оқиғаларын өнім санатымен байытып, содан кейін санат-аймақ түйіршіктілігінде агрегаттайды. `dim_products` ішінде әр `product_id` үшін бір жол болғандықтан, Join сәйкес оқиға жолдарын көбейтпейді. `INNER JOIN` өлшем жолы жоқ өнім оқиғаларын әдейі алып тастайды.


In [ ]:
lab.sql("""
SELECT
    d.category,
    e.region,
    COUNT(*) AS purchase_events,
    SUM(e.revenue) AS purchase_revenue
FROM events_modelled e
INNER JOIN dim_products d
    ON e.product_id = d.product_id
WHERE e.event_time >= '2020-03-03 00:00:00'
  AND e.event_time <  '2020-03-04 00:00:00'
  AND e.event_type = 'purchase'
GROUP BY d.category, e.region
ORDER BY purchase_revenue DESC, d.category, e.region
LIMIT 12
""", title="Purchase revenue by product category and region");


<a id="unit-u.34e1632c667f42ceb7245bab4fca941a"></a>

**Күтілетін нәтиже:** он екі санат-аймақ тобы түсімнің кему ретімен көрсетіледі. Бірінші жол — `region_08` ішіндегі `category_2`, 321 сатып алу оқиғасы және 128,004.16 түсім; он екінші — `region_07` ішіндегі `category_2`, 320 оқиға және 113,682.92. INNER JOIN тек сәйкес өлшем жолы бар оқиғаларды сақтағандықтан, ешбір санат `NULL` емес. Нәтиже түйіршіктілігі енді әр бастапқы оқиғаға бір жол емес, әр `category + region` тіркесіне бір жол.

<a id="unit-u.899b6706fdcf4b62a60df30bf2c79448"></a>

<a id="4-predict-how-duplicate-join-keys-change-the-result-grain"></a>

## 4. Қайталанатын Join кілттерінің нәтиже түйіршіктілігін қалай өзгертетінін болжау

Бақыланатын оқиға кестесінде төрт бастапқы жол бар:

| `event_id` | `product_id` | Жол дайындаған байланыс |
|---:|---:|---|
| `1` | `10` | Бір сәйкес өнім жолы |
| `2` | `20` | Екі сәйкес өнім-нұсқа жолы |
| `3` | `30` | Сәйкес өнім жолы жоқ |
| `4` | `NULL` | Join кілті жоқ |

Бақыланатын өнім кестесінде бес жол бар:

| `product_id` | `version_label` | Жол дайындаған байланыс |
|---:|---|---|
| `10` | `current` | `1` оқиғасына бір рет сәйкес |
| `20` | `old` | `2` оқиғасының бірінші сәйкестігі |
| `20` | `current` | `2` оқиғасының екінші сәйкестігі |
| `40` | `unused` | Оқиға жолы жоқ |
| `NULL` | `unknown` | Келесі бөлімде NULL сәйкестігін тексеруге қолданылады |

Join әр сәйкес жұпты қайтарады және оң жақ Join кілтінің бірегей болуын талап етпейді. `product_id = 20` бір рет `version_label = 'old'` және бір рет `version_label = 'current'` арқылы кездесетіндіктен, `2` оқиғасы екі өнім жолына да сәйкес келіп, екі нәтиже жолын береді.

Бұл жол көбеюі — жарамды Join әрекеті. Қажеттілігі күтілетін нәтиже түйіршіктілігіне байланысты: нәтиже әр оқиға–өнім-нұсқа байланысын қамтуы тиіс болса дұрыс, ал әр оқиғаға дәл бір байытылған жол қажет болса, алдымен шешілуі тиіс.


In [ ]:
lab.execute("DROP TABLE IF EXISTS join_event_cases")
lab.execute("DROP TABLE IF EXISTS join_product_cases")
lab.execute("""
CREATE TABLE join_event_cases (event_id BIGINT, product_id BIGINT NULL)
DUPLICATE KEY(event_id)
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.execute("""
CREATE TABLE join_product_cases (product_id BIGINT NULL, version_label VARCHAR(16))
DUPLICATE KEY(product_id)
DISTRIBUTED BY HASH(product_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("INSERT INTO join_event_cases VALUES (1, 10), (2, 20), (3, 30), (4, NULL)", title="Write controlled event rows")
lab.insert("INSERT INTO join_product_cases VALUES (10, 'current'), (20, 'old'), (20, 'current'), (40, 'unused'), (NULL, 'unknown')", title="Write controlled product rows")

lab.sql("""
SELECT e.event_id, e.product_id, d.version_label
FROM join_event_cases e
LEFT OUTER JOIN join_product_cases d
    ON e.product_id = d.product_id
ORDER BY e.event_id, d.version_label
""", title="One-to-many Join result");


<a id="unit-u.5f71aa52c4c845d98ca91f4dfe5d946c"></a>

**Күтілетін нәтиже:** төрт бастапқы оқиға жолы бес сәйкестік нәтиже жолын береді. `2` оқиғасы екі рет пайда болады, өйткені оның `product_id = 20` мәні `old` және `current` екеуіне де сәйкес келеді. Бұл LEFT OUTER JOIN болғандықтан, `3` және `4` оқиғалары қалады, бірақ олардың `version_label` мәні `NULL`. Қажетті нәтиже түйіршіктілігі әр оқиғаға бір жол болса, алдымен өнім кірісін әр `product_id` үшін бір жолға азайту керек; әр өнім нұсқасы қажет болса, бес жолдық нәтиже дұрыс.


<a id="unit-u.76b7bcd4dac44c37acdbd1fb9e448718"></a>

<a id="5-choose-whether-null-join-keys-can-match"></a>

## 5. NULL Join кілттерінің сәйкесе алатынын таңдау

Қалыпты теңдік екі операндтың бірі `NULL` болғанда unknown қайтарады, сондықтан `NULL = NULL` сәйкес жұп құрмайды. Doris NULL-қа қауіпсіз `<=>` теңдік операторын да береді; екі операнд та `NULL` болса, true қайтарады. Оны бизнес байланысы екі жетіспейтін кілтті бір мән деп анықтағанда ғана пайдаланыңыз.


In [ ]:
lab.sql("""
SELECT join_condition, matching_pairs
FROM (
    SELECT 1 AS sequence, '=' AS join_condition, COUNT(*) AS matching_pairs
    FROM join_event_cases e INNER JOIN join_product_cases d
        ON e.product_id = d.product_id
    WHERE e.event_id = 4
    UNION ALL
    SELECT 2, '<=>', COUNT(*)
    FROM join_event_cases e INNER JOIN join_product_cases d
        ON e.product_id <=> d.product_id
    WHERE e.event_id = 4
) comparison
ORDER BY sequence
""", title="NULL Join-key comparison");


<a id="unit-u.782360903c604689a35ec55d0e07af4d"></a>

**Күтілетін нәтиже:** `=` нөл сәйкес жұп, ал `<=>` бір жұп береді. NULL-aware Anti Join `NULL` айқын өңделуі тиіс байланысты `NOT IN`/Anti-Join жағдайын шешеді; ол мұнда тағы бір үлкен кесте сұрауын қоспай, сабақта таныстырылады.


<a id="unit-u.a7878bcc30c24ced833e49501bc37df9"></a>

<a id="6-connect-the-join-condition-to-the-physical-implementation"></a>

## 6. Join шартын физикалық іске асырумен байланыстыру

Теңдік шарты үшін Doris оң жақтан hash кестесін құрып, оны сол жақ жолдарымен тексере алады. Таза теңдік емес шарт hash кілті бола алмайды, сондықтан Doris Nested Loop Join таңдай алады. Екі мысал да тек шағын бақыланатын кестелерді пайдаланады; толық жоспарлар қорытынды астындағы ашылмалы бөлімде қалады.


In [ ]:
lab.compare_join_plans([
    ("Equi-condition", """
     EXPLAIN SHAPE PLAN
     SELECT COUNT(*)
     FROM join_event_cases e
     INNER JOIN join_product_cases d ON e.product_id = d.product_id
     """),
    ("Non-equi condition", """
     EXPLAIN SHAPE PLAN
     SELECT COUNT(*)
     FROM join_event_cases e
     INNER JOIN join_product_cases d ON e.product_id > d.product_id
     """),
], title="Join condition and physical implementation");


<a id="unit-u.70a672eafb4c4fdb97539444fc89dc28"></a>

**Күтілетін нәтиже:** теңдік шарты Hash Join, ал теңдік емес шарт Nested Loop Join пайдаланады. Hash Join ішінде оң жақ — құру жағы, сол жақ — тексеру жағы. Nested Loop Join жалпылама, бірақ әлдеқайда көп жол жұбын салыстыруы мүмкін, сондықтан зертханалық жұмыс кірісін әдейі шағын ұстайды.


<a id="unit-u.224b5a481b934a90bf61aeba4a1468bb"></a>

<a id="7-read-data-movement-from-the-plan-and-filtering-from-the-profile"></a>

## 7. Жоспардан дерек тасымалын, Profile ішінен сүзуді оқу

Broadcast және Shuffle бірнеше BE бар үлестірілген кластерде ең маңызды, онда олар Join деректерінің BE түйіндері арасында қалай жылжитынын анықтайды. Broadcast кіші кірісті Join орындайтын BE түйіндеріне көшіреді, ал Shuffle жолдарды Join кілті бойынша қайта таратады.

Бұл зертханалық жұмыс бір BE сынақ ортасын пайдаланады, сондықтан нақты BE аралық Shuffle жасамайды. Ол оңтайландырғыш жасаған тарату стратегиясы мен Exchange құрылымын көрсетеді. Бірінші жоспар салыстырмалы шағын `dim_products` қатынасы үшін оңтайландырғышқа Broadcast таңдауға мүмкіндік береді. Екінші жоспар `[shuffle]` hint құралын салыстыру жоспары ретінде әдейі пайдаланады; бұл әдепкі ұсыныс емес. `dim_products` Join кілті бойынша Hash bucket-ке бөлінгендіктен, сынақ ортасы сол bucket орналасуын сақтап, басқа кірісті қайта тарата отырып Bucket Shuffle көрсете алады. Өткен уақытты немесе желі өнімділігін емес, жоспар дәлелін салыстырыңыз.

Hash Join үшін Doris құру жағының өнім идентификаторларынан Runtime Filter жасап, оны тексеру жағы Scan операторына қолдана алады. Бұл сәйкес келе алмайтын оқиға жолдарын Join операторына жетпей тұрып қабылдамайды.


In [ ]:
lab.explain_plan("""
EXPLAIN SHAPE PLAN
SELECT COUNT(*)
FROM events_modelled e
INNER JOIN dim_products d ON e.product_id = d.product_id
WHERE d.category = 'category_1'
  AND e.event_time >= '2020-03-01 00:00:00'
  AND e.event_time <  '2020-03-02 00:00:00'
""", title="Default Join plan")

lab.explain_plan("""
EXPLAIN SHAPE PLAN
SELECT COUNT(*)
FROM events_modelled e
INNER JOIN [shuffle] dim_products d ON e.product_id = d.product_id
WHERE d.category = 'category_1'
  AND e.event_time >= '2020-03-01 00:00:00'
  AND e.event_time <  '2020-03-02 00:00:00'
""", title="Join plan with the [shuffle] hint");


<a id="unit-u.712efe61c29348eba7f8cdd24fca10e3"></a>

**Күтілетін нәтиже:** екі өңделмеген жоспар ағашын Join операторынан Scan операторларына қарай оқыңыз:

1. **Default Join plan** ішінен `hashJoin[INNER_JOIN broadcast]` табыңыз. Бұл кіші кірісі Broadcast болатын Inner Hash Join анықтайды.
2. Сол Hash Join жолынан `build RFs:RF... product_id->[product_id]` табыңыз. Doris құру жағының `product_id` мәндерінен бір немесе бірнеше Runtime Filter құрады. `RF0` сияқты Runtime Filter идентификаторлары өзгеруі мүмкін.
3. `events_modelled` тармағы астынан `PhysicalOlapScan[events_modelled] apply RFs: RF...` табыңыз. Тексеру жағы Scan сол Runtime Filter жазбаларын алады, сондықтан сәйкес келе алмайтын жолдар Join операторына жетпей тұрып қабылданбауы мүмкін.
4. **Join plan with the [shuffle] hint** ішінен `hashJoin[INNER_JOIN shuffleBucket]` табыңыз. Сыналған Doris 4.1.3 ортасында Broadcast емес жоспар `product_id` бойынша бар Hash bucket орналасуын қайта пайдаланып, Bucket Shuffle көрсетеді.
5. Hint бар жоспардың соңында `Hint log` `[shuffle]` мәнін `Used` астында тізіп, Doris hint қабылдағанын растауы тиіс.

Жалпы Partition Shuffle екі кірісті де Join кілтімен қайта таратады, ал Bucket Shuffle бір үйлесімді Bucket орналасуын сақтап, екінші кірісті қайта таратады. Екі кесте де сол бірге орналастыру шартына сай болса, Colocate Join shuffle тасымалын болдырмай алады. `EXPLAIN SHAPE PLAN` үлестірілген сұрау жоспарын сұрауды орындамай жасайтындықтан, мұнда өткен уақыт жоқ. Келесі ұяшық әдепкі Join орындап, оқиға жолдарының Join амалына жетпей сүзілгенін көру үшін Query Profile оқиды. Осы бір түйінді ортадағы өткен уақыт көп түйінді Join стратегияларына орын беруге қолданылмауы тиіс.

<a id="unit-u.10c79cb82f7f40048ea84297b984a1b6"></a>

<a id="execute-the-join-and-inspect-probe-side-filtering"></a>

### Join орындап, тексеру жағындағы сүзуді тексеру

`EXPLAIN SHAPE PLAN` көзделген Runtime Filter жазбаларын сипаттайды, бірақ сұрауды орындамайды. Сол әдепкі Join орындап, нақты Query Profile тексеріңіз. Төмендегі нәтиже — қайта пішімделген метрика кестесі емес, қысқартылған **өңделмеген MergedProfile үзіндісі**: ол Join `ProbeRows` және `events_modelled` Scan `RF... InputRows`, `RF... FilterRows`, `RowsProduced`, `ScanRows` санағыштарын сақтайды. Doris бірнеше Runtime Filter жасауы мүмкін, сондықтан әр сүзгінің өз санағыштарын тексеріңіз.


In [ ]:
lab.show_join_runtime_filter_profile("""
SELECT COUNT(*)
FROM events_modelled e
INNER JOIN dim_products d ON e.product_id = d.product_id
WHERE d.category = 'category_1'
  AND e.event_time >= '2020-03-01 00:00:00'
  AND e.event_time <  '2020-03-02 00:00:00'
""", probe_table="events_modelled", title="Probe-side Join Profile excerpt");

<a id="unit-u.b65b1442a1d1480f8815a2407e926e99"></a>

**Күтілетін нәтиже:** сұрау сәйкес оқиға санын қайтарады, одан кейін `HASH_JOIN_OPERATOR` және `OLAP_SCAN_OPERATOR(...table_name=events_modelled...)` тақырыптары бар қысқа мәтін үзіндісі беріледі.

Орындау даналары бойынша **`sum`** мәндерін мына ретпен оқыңыз:

1. `events_modelled` Scan астында `RF... InputRows` сол нақты Runtime Filter сүзгісіне ұсынылған жолдарды, ал `RF... FilterRows` оның қабылдамаған жолдарын санайды. Екі санағышты RF идентификаторы арқылы сәйкестендіріңіз; әртүрлі сүзгілердің кіріс сандарын қоспаңыз, өйткені сүзгілер тізбектей орындалуы мүмкін.
2. Осы тексеру жағы Scan операторындағы оң `RF... FilterRows` — кей оқиға жолдарының Join алдында қабылданбағанының орындалу дәлелі. Сыналған Doris 4.1.3 ортасында бір сүзгі 75,259 кіріс оқиға жолының 61,934 жолын, ал екіншісі 0 жолды қабылдамады; RF идентификаторлары мен сандары әр орындауда не ортада өзгеше болуы мүмкін.
3. `events_modelled` Scan астындағы `RowsProduced` — әрі қарай өткізген жол саны. Оны `HASH_JOIN_OPERATOR` астындағы `ProbeRows` мәнімен салыстырыңыз. Сыналған орындауда екеуі де 13,325 болды, бұл сұрау нәтижесіне сәйкес, өйткені қалған әр өнім идентификаторы бір өлшем жолына сәйкес келді.
4. `ScanRows` тексеру жағы Scan сканерлеген жолдарды санайды. Сыналған орындауда ол 75,259 болды, сондықтан `RF... FilterRows` мәнін Doris сақтау орнынан мүлде оқымаған жолдар саны ретінде оқуға **болмайды**. Бұл Profile нақты үнемделген диск енгізу/шығару көлемін емес, Join амалына азырақ жол жеткенін көрсетеді.

Сұрау нәтижесін кесте деректері мен SQL анықтайды. Runtime Filter идентификаторлары, типі, уақыты және әр сүзгі саны — орындау егжей-тегжейлері, сондықтан үлгі сандарын әмбебап күтілетін мән деп қабылдамай, көрсетілген Profile пайдаланыңыз.


<a id="unit-u.4308393f10ed4bccb478b978f1cbc655"></a>

<a id="8-practice-move-a-condition-from-on-to-where"></a>

## 8. Жаттығу: шартты ON ішінен WHERE ішіне көшіру

4 және 5-бөлімдердің екі шағын сынақ жиынын пайдаланыңыз: `join_event_cases` ішінде `product_id` мәндері 10, 20, 30 және `NULL` болатын төрт оқиға бар, ал `join_product_cases` ішінде 10-өнім (`current`), екі рет 20-өнім (`current` және `old`), 40-өнім және `NULL` кілті бар. Осы үш сұраудың әрқайсысы қанша жол қайтаратынын және қай оқиғаларды сақтайтынын болжаңыз:

1. `join_event_cases e LEFT JOIN join_product_cases p ON e.product_id = p.product_id`
2. сол Join, `ON` ішіне `AND p.version_label = 'current'` қосылған
3. 1-сұраудың Join амалынан кейін `WHERE p.version_label = 'current'`

Содан кейін әрқайсысында `e.event_id`, `e.product_id` және `p.version_label` бар үш сұрауды орындаңыз.

In [ ]:
# Your code here

<a id="unit-u.c51b35e4d7ec4d289a82134662857a32"></a>

Келесі ұяшық — үлгі шешім; ол жиналған. Салыстыру үшін оны ашыңыз немесе орындаңыз.

In [ ]:
practice_join = """
SELECT e.event_id, e.product_id, p.version_label
FROM join_event_cases AS e
LEFT JOIN join_product_cases AS p
  ON e.product_id = p.product_id{on_extra}
{where}ORDER BY e.event_id, p.version_label
"""
key_only = lab.sql(practice_join.format(on_extra="", where=""),
                   title="1 · LEFT JOIN on the key")
in_on = lab.sql(practice_join.format(on_extra="\n AND p.version_label = 'current'", where=""),
                title="2 · The version condition in ON")
in_where = lab.sql(practice_join.format(on_extra="", where="WHERE p.version_label = 'current'\n"),
                   title="3 · The version condition in WHERE")
assert (len(key_only), len(in_on), len(in_where)) == (5, 4, 2);

<a id="unit-u.447002eb68f54ce49d88a4d7d99107bd"></a>

**Күтілетін нәтиже**

| Сұрау | Жолдар | Не болды |
|---|---:|---|
| 1 · тек кілт | 5 | 2-оқиға 20-өнімнің екі нұсқасына да сәйкес, сондықтан екі рет көрінеді; 3 және 4 оқиғалары сәйкес емес және `NULL` өнім бағандарын сақтайды |
| 2 · шарт `ON` ішінде | 4 | Тек `current` жолдары сәйкес келе алады: 2-оқиға бір рет көрінеді, ал 3 және 4 оқиғалары әлі сақталады |
| 3 · шарт `WHERE` ішінде | 2 | Сүзгі Join кейін орындалады; `NULL` `'current'` емес, сондықтан сәйкес емес 3 және 4 оқиғалары Inner Join сияқты жоғалады |

`ON` ішінде оң жақ шарты қай жолдар сәйкес келе алатынын шешіп, әр сол жолды сақтайды. `WHERE` ішінде сол шарт Outer Join сақтаған жолдарды алып тастайды, сондықтан 3-сұрау «әр оқиға, бар болса ағымдағы өнімімен» емес, «ағымдағы өнімі бар оқиғалар» дегенге жауап береді.

<a id="unit-u.6b4d309cff8a48c6a28f13522ea4bcf5"></a>

<a id="stop-the-doris-sandbox"></a>

### Doris сынақ ортасын тоқтату

CPU мен жадты босату үшін осы қосымша ұяшықты орындаңыз. Docker атаулы томдары `events_modelled`, `dim_products` және бақыланатын Join кестелерін сақтайды.


In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris-rta
docker inspect --format 'container={{.State.Status}}' doris-rta
""", title="Stop the Doris sandbox");


<a id="unit-u.bff3b0fd2f1448cb9781ba3ae501f85d"></a>

**Күтілетін нәтиже:** Docker `container=exited` деп хабарлайды.

<a id="restart-the-doris-sandbox"></a>

### Doris сынақ ортасын қайта іске қосу

Басқа модульге өтпес бұрын осы ұяшықты орындаңыз. Ол бар контейнерді іске қосып, FE түйініне қайта қосылады және сақталған өнім өлшемін тексереді.

Бұл қайта іске қосу ұяшығы идемпотентті: оны Docker Desktop пен контейнер тоқтаған, іске қосылып жатқан немесе жұмыс істеп тұрған кезде орындауға болады. macOS жүйесінде қажет болса Docker Desktop ашылады; Linux жүйесінде ұяшықты орындамас бұрын Docker Engine іске қосыңыз.


In [ ]:
lab.start_container()

lab.connect()
lab.execute("USE doris_course")
lab.sql("SELECT COUNT(*) AS dimension_rows FROM dim_products", title="Recovered product dimension");


<a id="unit-u.b2458200ba6d4d12bce484da5c4161a8"></a>

**Күтілетін нәтиже:** контейнер `healthy` күйіне оралады және `dimension_rows` — 204,231.

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

Бастапқы идентификаторлардан детерминді өлшем жасадыңыз, жол сақтау талаптарынан Join түрлерін таңдадыңыз, жол көбеюін байқадыңыз, NULL сәйкестігін басқардыңыз, Hash Join мен Nested Loop Join ажыраттыңыз, жоспарланған Join таратуын оқып, тексеру жағындағы Runtime Filter сандарын байқадыңыз және факт-өлшем талдамалық нәтижесін жасадыңыз.

Ресми анықтамалар: [Doris Join амалдары](https://doris.apache.org/docs/4.x/query-data/join/) · [EXPLAIN](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-query/EXPLAIN/) · [Runtime Filter](https://doris.apache.org/docs/4.x/query-acceleration/optimization-technology-principle/runtime-filter/) · [Join Shuffle режимін реттеу](https://doris.apache.org/docs/4.x/query-acceleration/tuning/tuning-plan/adjusting-join-shuffle/)


<a id="unit-u.6243b816e7f54ea1a8b40abca80c0194"></a>

<a id="next"></a>

## Келесі қадам

[6-тест: Apache Doris жүйесінде деректерді біріктіру](quiz06_join_semantics_and_execution.ipynb) ашыңыз, содан кейін [7-модуль: Apache Doris жүйесінде деректерді жаңарту және жою](../module07-updating-deleting/course.md) бөліміне өтіңіз.
